# LeetCode 537 — Complex Number Multiplication bằng Python

Link đề: https://leetcode.com/problems/complex-number-multiplication/ · Độ khó: **Medium** · Chủ đề: Math, String, Simulation

Skeleton LeetCode:

```python
class Solution:
    def complexNumberMultiply(self, num1: str, num2: str) -> str:
        ...
```

Ghi chú: các ví dụ 1-2 lấy từ đề; những ví dụ khác do notebook tự đặt (đáp án đã kiểm chứng bằng code), cần thì đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho hai số phức dạng chuỗi `"a+bi"` (ví dụ `"1+-1i"`), nhân chúng và trả về kết quả **cũng dưới dạng chuỗi** `"real+imaginaryi"`.
2. Nắm các công cụ Python cần dùng: cắt chuỗi (`s[:-1]`, `rstrip`), `str.split("+")`, `int("-1")`, f-string, gán nhiều biến cùng lúc (tuple unpacking), `re.fullmatch` với nhóm `( )`, kiểu `complex` có sẵn của Python, và nhân ma trận `2 x 2` bằng list lồng nhau.
3. Đi từ **khai triển đa thức rồi rút gọn lũy thừa của `i`** -> **công thức `(ac - bd) + (ad + bc)i`** -> **mẹo Gauss chỉ dùng 3 phép nhân** -> **biểu diễn số phức bằng ma trận `2 x 2`** -> **dùng kiểu `complex` có sẵn** (với regex để đọc chuỗi).
4. Hiểu hai mấu chốt: (a) vì `i^2 = -1` nên số hạng `bi * di` biến thành `-bd` ở phần thực; (b) định dạng đầu ra **luôn** là `thực + ảoi` với dấu `+` ở giữa kể cả khi phần ảo âm (`"0+-2i"`).
5. Tránh các lỗi kinh điển: quên `i^2 = -1`, quên cắt chữ `i` cuối chuỗi, in sai định dạng (`"0-2i"`), dùng số thực dấu phẩy động (hình học cực) làm kết quả bị lệch, và gọi `complex("1+-1j")` trực tiếp.

## 2. Đề bài & ví dụ

Một **số phức** có thể được biểu diễn bằng chuỗi dạng `"real+imaginaryi"` trong đó:

- `real` là phần thực, là số nguyên trong khoảng `[-100, 100]`.
- `imaginary` là phần ảo, là số nguyên trong khoảng `[-100, 100]`.
- `i^2 == -1`.

Cho hai số phức `num1` và `num2` ở dạng chuỗi, hãy trả về **chuỗi** biểu diễn **tích** của chúng (cũng theo dạng `"real+imaginaryi"`).

**Ví dụ 1**

- Input: `num1 = "1+1i"`, `num2 = "1+1i"`
- Output: `"0+2i"`
- Giải thích: `(1 + i) * (1 + i) = 1 + i^2 + 2i = 2i`, và cần viết lại theo dạng `0+2i`.

**Ví dụ 2**

- Input: `num1 = "1+-1i"`, `num2 = "1+-1i"`
- Output: `"0+-2i"`
- Giải thích: `(1 - i) * (1 - i) = 1 + i^2 - 2i = -2i`, viết lại thành `0+-2i`.

**Ràng buộc:** `num1` và `num2` là các số phức hợp lệ (mỗi phần thực/ảo là số nguyên trong `[-100, 100]`).

Chú ý quan trọng: dấu `+` luôn nằm **giữa** phần thực và phần ảo, ngay cả khi phần ảo âm: `"1+-1i"` chứ không phải `"1-1i"`. Kết quả cũng viết đúng như vậy, `"0+-2i"`.

In [1]:
import re
import random
import time
import math
import cmath
from fractions import Fraction

examples = [
    (("1+1i", "1+1i"), "0+2i"),
    (("1+-1i", "1+-1i"), "0+-2i"),
]
for (a, b), expected in examples:
    print(f"{a!r} * {b!r} -> kỳ vọng {expected!r}")

'1+1i' * '1+1i' -> kỳ vọng '0+2i'
'1+-1i' * '1+-1i' -> kỳ vọng '0+-2i'


## 3. Kiến thức nền

### 3.1. Số phức và đơn vị ảo `i`

Số phức có dạng `a + bi`: `a` là **phần thực**, `b` là **phần ảo**, và `i` là "đơn vị ảo" với tính chất duy nhất cần nhớ: **`i * i = -1`**. Từ đó `i^3 = i^2 * i = -i` và `i^4 = 1`: lũy thừa của `i` lặp lại theo chu kỳ **4**: `1, i, -1, -i, 1, i, ...`.

Python có sẵn kiểu số phức, nhưng dùng chữ **`j`** thay vì `i`: `1+1j`. Ta dùng nó để **kiểm chứng** thôi.

In [2]:
print(1j * 1j)                 # (-1+0j): đúng là i*i = -1
print([1j ** k for k in range(8)])      # chu kỳ 4
z = (1 + 1j) * (1 + 1j)
print(z, z.real, z.imag)

(-1+0j)
[(1+0j), 1j, (-1+0j), (-0-1j), (1+0j), 1j, (-1+0j), (-0-1j)]
2j 0.0 2.0


### 3.2. Nhân hai số phức = nhân hai "ngoặc" (phân phối)

`(a + bi)(c + di)` khai triển như nhân hai đa thức: lấy **mỗi số hạng của ngoặc đầu nhân mỗi số hạng của ngoặc sau** (4 tích):

```
(a + bi)(c + di) = ac + adi + bci + bdi^2
```

Vì `i^2 = -1` nên `bdi^2 = -bd`. Gộp phần thực và phần ảo:

```
= (ac - bd) + (ad + bc)i
```

Đây là **công thức nhân số phức**, bài này chỉ là "đọc chuỗi, áp công thức, in chuỗi". Thử số: `(1+1i)(1+1i)`: `a=b=c=d=1` -> phần thực `1*1 - 1*1 = 0`, phần ảo `1*1 + 1*1 = 2` -> `0+2i`. Khớp ví dụ 1.

In [3]:
a, b, c, d = 1, 1, 1, 1
print("thực:", a * c - b * d, "| ảo:", a * d + b * c)

a, b, c, d = 1, -1, 1, -1          # ví dụ 2: (1 - i)(1 - i)
print("thực:", a * c - b * d, "| ảo:", a * d + b * c)

thực: 0 | ảo: 2
thực: 0 | ảo: -2


### 3.3. Cắt chuỗi: `s[:-1]`, `rstrip`, `split`, `int`

- `s[:-1]` là chuỗi bỏ **ký tự cuối**. `"1+-1i"[:-1]` = `"1+-1"`.
- `s.rstrip("i")` bỏ **mọi** ký tự `i` ở cuối chuỗi (ở đây chỉ có một).
- `s.split("+")` cắt chuỗi tại mỗi dấu `+` thành một **list**: `"1+-1".split("+")` = `["1", "-1"]`.
- `int("-1")` = `-1`; `int("1i")` thì **lỗi** `ValueError` (nên phải bỏ `i` trước).

Chú ý vì sao `split("+")` an toàn: trong chuỗi hợp lệ, dấu `+` **chỉ** xuất hiện đúng một lần, làm dấu ngăn giữa phần thực và phần ảo. Dấu `-` chỉ là dấu âm nằm **trong** từng số nên không bị tách nhầm.

In [4]:
s = "1+-1i"
print(s[:-1])
print(s.rstrip("i"))
print(s[:-1].split("+"))
print("-3+-2i"[:-1].split("+"))       # phần thực âm cũng không sao
print(int("-1"), int("100"), int("0"))
try:
    int("1i")
except ValueError as e:
    print("int('1i') ->", type(e).__name__)

1+-1
1+-1
['1', '-1']
['-3', '-2']
-1 100 0
int('1i') -> ValueError


### 3.4. Gán nhiều biến cùng lúc (tuple unpacking) và hàm `parse`

Nếu một list có đúng 2 phần tử thì `x, y = lst` gán `x = lst[0]`, `y = lst[1]`. Ta gói việc "đọc chuỗi thành hai số nguyên" vào một hàm `parse` trả về cặp `(thực, ảo)`.

In [5]:
x, y = ["3", "-4"]
print(x, y)

def parse(s):
    """"a+bi" -> (a, b) là hai số nguyên"""
    real, imag = s[:-1].split("+")
    return int(real), int(imag)

print(parse("1+-1i"), parse("-100+100i"), parse("0+0i"))

3 -4
(1, -1) (-100, 100) (0, 0)


### 3.5. f-string và định dạng đầu ra

`f"{r}+{i}i"` ghép số vào chuỗi. Số âm tự có dấu `-`: `f"{0}+{-2}i"` = `"0+-2i"`, đúng định dạng đề yêu cầu (dấu `+` luôn có, bất kể phần ảo âm hay không).

In [6]:
r, i = 0, -2
print(f"{r}+{i}i")
print(f"{-5}+{7}i", f"{0}+{0}i")

0+-2i
-5+7i 0+0i


### 3.6. Biểu thức chính quy (regex) mức nhập môn: `re.fullmatch` và nhóm

Regex là "mẫu" để mô tả chuỗi. Ta cần mẫu `(-?\d+)\+(-?\d+)i`:

- `-?` : dấu `-` có thể có hoặc không. `\d+` : một hay nhiều chữ số.
- `\+` : dấu `+` thật (vì `+` có nghĩa đặc biệt trong regex nên phải thêm `\`).
- Dấu ngoặc tròn `( )` tạo **nhóm** để lấy ra phần khớp: `m.group(1)`, `m.group(2)`.
- `re.fullmatch(mẫu, chuỗi)` yêu cầu khớp **toàn bộ** chuỗi, trả `None` nếu không khớp.

In [7]:
pattern = re.compile(r"(-?\d+)\+(-?\d+)i")
m = pattern.fullmatch("1+-1i")
print(m.group(1), m.group(2), m.groups())
print(pattern.fullmatch("1-1i"))          # None: sai định dạng của đề
print(pattern.fullmatch("-100+-100i").groups())

1 -1 ('1', '-1')
None
('-100', '-100')


### 3.7. Nhân ma trận `2 x 2` bằng list lồng nhau

Ma trận `2 x 2` lưu bằng list của hai list: `M = [[p, q], [r, s]]`. Nhân hai ma trận `A * B`: ô `(hàng i, cột j)` = tổng `A[i][k] * B[k][j]` theo `k`. Ta sẽ dùng nó ở cách 4.

In [8]:
def matmul2(A, B):
    return [[sum(A[i][k] * B[k][j] for k in range(2)) for j in range(2)] for i in range(2)]

A = [[1, 2], [3, 4]]
B = [[0, 1], [1, 0]]
print(matmul2(A, B))
print(matmul2([[1, 0], [0, 1]], A))   # nhân ma trận đơn vị: không đổi

[[2, 1], [4, 3]]
[[1, 2], [3, 4]]


### 3.8. Kiểu `complex` của Python và vì sao float cần cẩn thận

`complex(a, b)` tạo số phức `a + bj`; `.real`, `.imag` trả về **float** (số thực dấu phẩy động). Với số nguyên nhỏ (ở đây tối đa cỡ `20000`) phép nhân/cộng float vẫn **chính xác tuyệt đối**, nhưng nếu đi qua `sin`, `cos`, `sqrt` thì sẽ có **sai số** (mục 13). Python **không** đọc được chuỗi `"1+-1j"` (hai dấu liền nhau) nên vẫn cần tự tách.

In [9]:
z = complex(3, -2) * complex(1, 4)
print(z, type(z.real), int(z.real), int(z.imag))
try:
    complex("1+-1j")
except ValueError as e:
    print("complex('1+-1j') ->", type(e).__name__)
print(complex("1-1j"))                    # chuỗi chuẩn của Python thì được

(11+10j) <class 'float'> 11 10
complex('1+-1j') -> ValueError
(1-1j)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Tính tay `(3 + 2i)(1 + -4i)`, tức `a=3, b=2, c=1, d=-4`. Dùng **bảng nhân 2 x 2** (mỗi ô là một trong 4 tích):

| `x` | `c = 1` | `di = -4i` |
|---|---|---|
| `a = 3` | `ac = 3` | `adi = -12i` |
| `bi = 2i` | `bci = 2i` | `bdi^2 = 2 * (-4) * i^2 = -8 * (-1) = +8` |

Gộp: phần **thực** = ô `ac` + ô `bd i^2` = `3 + 8 = 11`; phần **ảo** = `-12 + 2 = -10`. Kết quả `11 + -10i`, viết là `"11+-10i"`.

Quy trình cho máy tính chỉ có 3 bước:

1. **Đọc**: từ mỗi chuỗi lấy ra hai số nguyên `(a, b)` và `(c, d)`.
2. **Tính**: `real = a*c - b*d`, `imag = a*d + b*c`.
3. **Ghi**: ghép `f"{real}+{imag}i"`.

Bảng trạng thái nhanh cho ví dụ 2 (`"1+-1i" * "1+-1i"`):

| Bước | Giá trị |
|---|---|
| đọc | `a=1, b=-1, c=1, d=-1` |
| `ac` | `1` |
| `bd` | `(-1)(-1) = 1` |
| thực `= ac - bd` | `1 - 1 = 0` |
| `ad` | `1 * (-1) = -1` |
| `bc` | `(-1) * 1 = -1` |
| ảo `= ad + bc` | `-2` |
| kết quả | `"0+-2i"` |

## 5. Cách 1 — Brute Force / baseline: khai triển đa thức rồi rút gọn lũy thừa của `i`

**Ý tưởng.** Chưa dùng công thức có sẵn: ta xử lý số phức như **đa thức theo `i`**. Mỗi số phức là một từ điển `{lũy thừa: hệ số}`, ví dụ `1+-1i` -> `{0: 1, 1: -1}`. Nhân hai đa thức: với mỗi cặp số hạng `(p, x)` và `(q, y)` cộng `x * y` vào ô lũy thừa `p + q`. Sau đó **rút gọn** `i^k` về một trong `1, i, -1, -i` theo `k mod 4`:

| `k mod 4` | `i^k` | cộng vào |
|---|---|---|
| 0 | `+1` | phần thực |
| 1 | `+i` | phần ảo |
| 2 | `-1` | phần thực, đổi dấu |
| 3 | `-i` | phần ảo, đổi dấu |

Cách này "ngốc" hơn công thức nhưng **tự nhiên** nhất: chỉ là nhân hai ngoặc rồi dùng `i^2 = -1`.

In [10]:
def to_poly(s):
    a, b = parse(s)
    return {0: a, 1: b}

def multiply_poly(p1, p2):
    res = {}
    for pa, x in p1.items():
        for pb, y in p2.items():
            res[pa + pb] = res.get(pa + pb, 0) + x * y
    return res

def reduce_i(poly):
    real = imag = 0
    for k, coef in poly.items():
        r = k % 4
        if r == 0:
            real += coef
        elif r == 1:
            imag += coef
        elif r == 2:
            real -= coef
        else:
            imag -= coef
    return real, imag

def complex_multiply_poly(num1, num2):
    p = multiply_poly(to_poly(num1), to_poly(num2))
    real, imag = reduce_i(p)
    return f"{real}+{imag}i"

print(multiply_poly(to_poly("1+-1i"), to_poly("1+-1i")))       # {0: 1, 1: -2, 2: 1}
for (a, b), expected in examples:
    got = complex_multiply_poly(a, b)
    print(a, "*", b, "->", got, "OK" if got == expected else "SAI")

{0: 1, 1: -2, 2: 1}
1+1i * 1+1i -> 0+2i OK
1+-1i * 1+-1i -> 0+-2i OK


In [11]:
class Solution1:
    """Cách 1: khai triển đa thức rồi rút gọn lũy thừa của i."""
    def complexNumberMultiply(self, num1: str, num2: str) -> str:
        def poly(s):
            real, imag = s[:-1].split("+")
            return {0: int(real), 1: int(imag)}

        res = {}
        for pa, x in poly(num1).items():
            for pb, y in poly(num2).items():
                res[pa + pb] = res.get(pa + pb, 0) + x * y
        real = imag = 0
        for k, coef in res.items():
            r = k % 4
            if r == 0:
                real += coef
            elif r == 1:
                imag += coef
            elif r == 2:
                real -= coef
            else:
                imag -= coef
        return f"{real}+{imag}i"

print(Solution1().complexNumberMultiply("3+2i", "1+-4i"))     # 11+-10i (khớp tính tay ở mục 4)

11+-10i


**Độ phức tạp.** Mỗi số có đúng 2 số hạng nên chỉ có 4 tích: **Time `O(1)`**, Space `O(1)`. (Nếu số phức mở rộng thành đa thức bậc `k`, cách này sẽ là `O(k^2)`: đó là sức mạnh tổng quát của nó.)

**Nút thắt.** Tuy chạy `O(1)`, cách này **vòng vo**: dựng từ điển, vòng lặp lồng nhau, rút gọn bằng `k % 4`... trong khi ta biết trước chỉ có đúng 4 tích và `i^2 = -1`. Ta có thể **viết thẳng kết quả** bằng một công thức.

## 6. Ý tưởng thuật toán / bước đột phá

Vì mỗi số phức chỉ có hai số hạng (`a` và `bi`), mọi tích chỉ là một trong bốn:

```
a * c        -> thực
a * di       -> ảo
bi * c       -> ảo
bi * di      -> bd * i^2 = -bd -> thực (đổi dấu!)
```

Gộp phần thực: `ac - bd`. Gộp phần ảo: `ad + bc`. **Không cần đa thức, không cần chu kỳ của `i`**: chỉ cần công thức

```
(a + bi)(c + di) = (ac - bd) + (ad + bc)i
```

Từ đó có hai hướng mở rộng thú vị:

- **Giảm số phép nhân** (mẹo Gauss): dùng 3 phép nhân thay vì 4 (cách 3).
- **Đổi cách biểu diễn**: mỗi số phức là một ma trận `2 x 2`; nhân số phức = nhân ma trận (cách 4). Hoặc **nhờ thư viện**: dùng kiểu `complex` (cách 5).

Phần "khó" thật sự của bài chỉ là **đọc chuỗi** (đừng quên chữ `i` cuối và dấu `-` trong phần ảo) và **ghi đúng định dạng**.

## 7. Các cách tiếp theo

### Cách 2 — Công thức trực tiếp `(ac - bd) + (ad + bc)i`

**Giải thích từng đoạn.**

1. **Đọc:** `num1[:-1]` bỏ chữ `i` cuối, `.split("+")` tách thành hai chuỗi số, `int` đổi thành số nguyên: `a, b`. Tương tự `c, d`.
2. **Tính:** `real = a*c - b*d`, `imag = a*d + b*c`.
3. **Ghi:** `f"{real}+{imag}i"`.

Demo từng đoạn:

In [12]:
num1, num2 = "3+2i", "1+-4i"

# đoạn 1: đọc
a, b = map(int, num1[:-1].split("+"))
c, d = map(int, num2[:-1].split("+"))
print("a, b, c, d =", a, b, c, d)

# đoạn 2: tính
real = a * c - b * d
imag = a * d + b * c
print("real, imag =", real, imag)

# đoạn 3: ghi
print(f"{real}+{imag}i")

a, b, c, d = 3 2 1 -4
real, imag = 11 -10
11+-10i


Ở đoạn 1 ta dùng `map(int, list)`: áp `int` lên từng phần tử của list, và `a, b = ...` gán hai giá trị. Đây chỉ là cách viết gọn của `int(x)` cho mỗi phần tử.

**Code nộp LeetCode (cách 2):**

In [13]:
class Solution2:
    """Cách 2: công thức trực tiếp."""
    def complexNumberMultiply(self, num1: str, num2: str) -> str:
        a, b = map(int, num1[:-1].split("+"))
        c, d = map(int, num2[:-1].split("+"))
        return f"{a * c - b * d}+{a * d + b * c}i"

for (x, y), expected in examples:
    assert Solution2().complexNumberMultiply(x, y) == expected
print("cách 2 qua các ví dụ")

cách 2 qua các ví dụ


**Độ phức tạp (cách 2).** Chuỗi dài tối đa vài ký tự (số nguyên `[-100, 100]`) nên cả đọc, tính, ghi đều là hằng số: **Time `O(1)`, Space `O(1)`**. (Nếu coi độ dài chuỗi là `L` thì đọc/ghi là `O(L)`, nhưng `L` bị chặn nên vẫn là hằng.)

### Cách 3 — Mẹo Gauss: 3 phép nhân thay vì 4

**Giải thích.** Carl Friedrich Gauss nhận ra có thể tính tích số phức chỉ với **3 phép nhân** (đổi bớt một phép nhân lấy vài phép cộng):

```
k1 = c * (a + b)
k2 = a * (d - c)
k3 = b * (c + d)
real = k1 - k3          # = ac + bc - bc - bd = ac - bd
imag = k1 + k2          # = ac + bc + ad - ac = ad + bc
```

Kiểm tra bằng đại số: `k1 - k3 = ac + bc - bc - bd = ac - bd` (đúng phần thực), `k1 + k2 = ac + bc + ad - ac = ad + bc` (đúng phần ảo).

Với số nhỏ thế này mẹo không giúp gì về tốc độ, nhưng với **số rất lớn** (nhân số nguyên khổng lồ, nhân đa thức bằng thuật toán Karatsuba) việc bớt một phép nhân là rất quý. Ở đây nó là một cách nhìn **khác** về cùng một tích.

In [14]:
a, b, c, d = 3, 2, 1, -4
k1 = c * (a + b)
k2 = a * (d - c)
k3 = b * (c + d)
print("k1, k2, k3 =", k1, k2, k3)
print("real =", k1 - k3, "| imag =", k1 + k2)       # 11 và -10, giống tính tay

k1, k2, k3 = 5 -15 -6
real = 11 | imag = -10


**Code nộp LeetCode (cách 3):**

In [15]:
class Solution3:
    """Cách 3: mẹo Gauss, 3 phép nhân."""
    def complexNumberMultiply(self, num1: str, num2: str) -> str:
        a, b = map(int, num1[:-1].split("+"))
        c, d = map(int, num2[:-1].split("+"))
        k1 = c * (a + b)
        k2 = a * (d - c)
        k3 = b * (c + d)
        return f"{k1 - k3}+{k1 + k2}i"

for (x, y), expected in examples:
    assert Solution3().complexNumberMultiply(x, y) == expected
print("cách 3 qua các ví dụ")

cách 3 qua các ví dụ


**Độ phức tạp (cách 3).** `O(1)` thời gian và bộ nhớ. Số phép nhân: 3 (so với 4 của cách 2), đổi lại nhiều phép cộng trừ hơn.

### Cách 4 — Biểu diễn số phức bằng ma trận `2 x 2`

**Giải thích.** Gắn mỗi số phức `a + bi` với ma trận

```
        [ a  -b ]
M(z) =  [ b   a ]
```

Điều kỳ diệu: `M(z1) * M(z2) = M(z1 * z2)`, tức **nhân ma trận chính là nhân số phức**. Thật vậy, nhân hai ma trận `M(a,b)` và `M(c,d)`:

```
[ a -b ] [ c -d ]   [ ac - bd   -(ad + bc) ]
[ b  a ] [ d  c ] = [ ad + bc     ac - bd  ]
```

Ma trận kết quả vẫn có dạng `M(real, imag)` với `real = ac - bd`, `imag = ad + bc` (phần tử ở góc trên trái và dưới trái). Ta chỉ cần đọc **cột đầu**: `real = R[0][0]`, `imag = R[1][0]`. (Vì sao `i` tương ứng ma trận `[[0,-1],[1,0]]`: nhân nó hai lần cho `-I`, đúng `i^2 = -1` - một phép quay 90 độ.)

In [16]:
def mat_of(a, b):
    return [[a, -b], [b, a]]

M1, M2 = mat_of(3, 2), mat_of(1, -4)
R = matmul2(M1, M2)
print(R)
print("real =", R[0][0], "| imag =", R[1][0])
print("kiểm tra dạng M(real, imag):", R == mat_of(R[0][0], R[1][0]))

[[11, 10], [-10, 11]]
real = 11 | imag = -10
kiểm tra dạng M(real, imag): True


**Code nộp LeetCode (cách 4):**

In [17]:
class Solution4:
    """Cách 4: số phức = ma trận 2x2, nhân ma trận rồi đọc cột đầu."""
    def complexNumberMultiply(self, num1: str, num2: str) -> str:
        a, b = map(int, num1[:-1].split("+"))
        c, d = map(int, num2[:-1].split("+"))
        A = [[a, -b], [b, a]]
        B = [[c, -d], [d, c]]
        R = [[sum(A[i][k] * B[k][j] for k in range(2)) for j in range(2)] for i in range(2)]
        return f"{R[0][0]}+{R[1][0]}i"

for (x, y), expected in examples:
    assert Solution4().complexNumberMultiply(x, y) == expected
print("cách 4 qua các ví dụ")

cách 4 qua các ví dụ


**Độ phức tạp (cách 4).** Nhân hai ma trận `2 x 2`: 8 phép nhân, `O(1)` thời gian và bộ nhớ. Cách này thừa về tính toán nhưng cho thấy một **góc nhìn đại số**: số phức là một cấu trúc con của ma trận.

### Cách 5 — Dùng kiểu `complex` có sẵn (đọc chuỗi bằng regex)

**Giải thích.**

1. Dùng regex `(-?\d+)\+(-?\d+)i` để lấy hai nhóm số (mục 3.6); cách đọc này khác `split`.
2. Đổi thành hai số phức `complex(a, b)` và `complex(c, d)` rồi **nhân bằng toán tử `*` của Python**.
3. Lấy `.real`, `.imag` (kiểu float) và `int(...)` về số nguyên. Với số nguyên nhỏ, phép nhân float **chính xác tuyệt đối** nên `int` an toàn.

Đây là cách ngắn gọn khi ngôn ngữ đã hỗ trợ số phức, nhưng cần ý thức về float (chỉ an toàn khi không có `sin`/`cos`/`sqrt`).

In [18]:
pattern = re.compile(r"(-?\d+)\+(-?\d+)i")

def to_complex(s):
    a, b = map(int, pattern.fullmatch(s).groups())
    return complex(a, b)

z = to_complex("3+2i") * to_complex("1+-4i")
print(z, "->", f"{int(z.real)}+{int(z.imag)}i")

(11-10j) -> 11+-10i


**Code nộp LeetCode (cách 5):**

In [19]:
class Solution5:
    """Cách 5: regex + kiểu complex có sẵn của Python."""
    def complexNumberMultiply(self, num1: str, num2: str) -> str:
        pattern = r"(-?\d+)\+(-?\d+)i"
        a, b = map(int, re.fullmatch(pattern, num1).groups())
        c, d = map(int, re.fullmatch(pattern, num2).groups())
        z = complex(a, b) * complex(c, d)
        return f"{int(z.real)}+{int(z.imag)}i"

for (x, y), expected in examples:
    assert Solution5().complexNumberMultiply(x, y) == expected
print("cách 5 qua các ví dụ")

cách 5 qua các ví dụ


**Độ phức tạp (cách 5).** `O(1)` thời gian và bộ nhớ (regex trên chuỗi ngắn bị chặn). Thực tế chậm hơn cách 2 vì phải biên dịch/khớp regex.

**Tóm tắt năm cách:** (1) khai triển đa thức + rút gọn `i^k`; (2) công thức trực tiếp; (3) mẹo Gauss 3 phép nhân; (4) ma trận `2 x 2`; (5) kiểu `complex` + regex. Cả năm đều `O(1)`; chúng khác nhau ở **cách nghĩ** chứ không ở tốc độ.

## 8. TRACE từng bước

### 8.1. Trace cách 2 (công thức) trên `"3+2i" * "1+-4i"`

Hàm in từng giai đoạn: chuỗi gốc -> bỏ `i` cuối -> tách theo `+` -> số nguyên -> từng tích -> kết quả.

| Cột | Ý nghĩa |
|---|---|
| `num[:-1]` | chuỗi sau khi bỏ chữ `i` cuối |
| `split` | list hai chuỗi số |
| `ac`, `bd`, `ad`, `bc` | bốn tích thành phần |

In [20]:
def trace_formula(num1, num2):
    print(f"num1 = {num1!r}, num2 = {num2!r}")
    parts = []
    for name, s in (("num1", num1), ("num2", num2)):
        stripped = s[:-1]
        halves = stripped.split("+")
        print(f"  {name}[:-1] = {stripped!r} -> split('+') = {halves}")
        parts.append(tuple(int(h) for h in halves))
    (a, b), (c, d) = parts
    print(f"  a={a}, b={b}, c={c}, d={d}")
    ac, bd, ad, bc = a * c, b * d, a * d, b * c
    print(f"  ac={ac}, bd={bd}, ad={ad}, bc={bc}")
    real, imag = ac - bd, ad + bc
    print(f"  thực = ac - bd = {ac} - ({bd}) = {real}")
    print(f"  ảo   = ad + bc = {ad} + ({bc}) = {imag}")
    out = f"{real}+{imag}i"
    print("  kết quả:", repr(out))
    return out

trace_formula("3+2i", "1+-4i")

num1 = '3+2i', num2 = '1+-4i'
  num1[:-1] = '3+2' -> split('+') = ['3', '2']
  num2[:-1] = '1+-4' -> split('+') = ['1', '-4']
  a=3, b=2, c=1, d=-4
  ac=3, bd=-8, ad=-12, bc=2
  thực = ac - bd = 3 - (-8) = 11
  ảo   = ad + bc = -12 + (2) = -10
  kết quả: '11+-10i'


'11+-10i'

In [21]:
trace_formula("1+-1i", "1+-1i")

num1 = '1+-1i', num2 = '1+-1i'
  num1[:-1] = '1+-1' -> split('+') = ['1', '-1']
  num2[:-1] = '1+-1' -> split('+') = ['1', '-1']
  a=1, b=-1, c=1, d=-1
  ac=1, bd=1, ad=-1, bc=-1
  thực = ac - bd = 1 - (1) = 0
  ảo   = ad + bc = -1 + (-1) = -2
  kết quả: '0+-2i'


'0+-2i'

Ở ví dụ 2: `bd = (-1)(-1) = +1` nên phần thực là `1 - 1 = 0`, còn `ad = bc = -1` nên phần ảo là `-2`. Đúng `"0+-2i"` của đề.

### 8.2. Trace cách 1 (đa thức + rút gọn `i^k`)

In từng cặp số hạng, lũy thừa của `i` và việc rút gọn theo `k mod 4`.

In [22]:
def trace_poly(num1, num2):
    p1, p2 = to_poly(num1), to_poly(num2)
    print("p1 =", p1, "| p2 =", p2)
    res = {}
    for pa, x in p1.items():
        for pb, y in p2.items():
            res[pa + pb] = res.get(pa + pb, 0) + x * y
            print(f"  ({x}) i^{pa} * ({y}) i^{pb} = {x * y} i^{pa + pb}   -> bảng lũy thừa: {res}")
    real = imag = 0
    for k, coef in sorted(res.items()):
        r = k % 4
        where = {0: "+thực", 1: "+ảo", 2: "-thực", 3: "-ảo"}[r]
        print(f"  rút gọn: {coef} * i^{k} (k mod 4 = {r}) -> {where}")
        if r == 0: real += coef
        elif r == 1: imag += coef
        elif r == 2: real -= coef
        else: imag -= coef
    print(f"  thực = {real}, ảo = {imag}")
    return f"{real}+{imag}i"

trace_poly("3+2i", "1+-4i")

p1 = {0: 3, 1: 2} | p2 = {0: 1, 1: -4}
  (3) i^0 * (1) i^0 = 3 i^0   -> bảng lũy thừa: {0: 3}
  (3) i^0 * (-4) i^1 = -12 i^1   -> bảng lũy thừa: {0: 3, 1: -12}
  (2) i^1 * (1) i^0 = 2 i^1   -> bảng lũy thừa: {0: 3, 1: -10}
  (2) i^1 * (-4) i^1 = -8 i^2   -> bảng lũy thừa: {0: 3, 1: -10, 2: -8}
  rút gọn: 3 * i^0 (k mod 4 = 0) -> +thực
  rút gọn: -10 * i^1 (k mod 4 = 1) -> +ảo
  rút gọn: -8 * i^2 (k mod 4 = 2) -> -thực
  thực = 11, ảo = -10


'11+-10i'

Số hạng `-8 * i^2`: `k = 2` nên `k mod 4 = 2` -> **trừ** `-8` khỏi phần thực -> `+8`. Đúng ô `bd i^2 = +8` trong bảng nhân ở mục 4.

### 8.3. Trace cách 3 (Gauss) và cách 4 (ma trận)

In [23]:
def trace_gauss(num1, num2):
    a, b = map(int, num1[:-1].split("+"))
    c, d = map(int, num2[:-1].split("+"))
    k1, k2, k3 = c * (a + b), a * (d - c), b * (c + d)
    print(f"k1 = c(a+b) = {c}*({a}+{b}) = {k1}")
    print(f"k2 = a(d-c) = {a}*({d}-{c}) = {k2}")
    print(f"k3 = b(c+d) = {b}*({c}+{d}) = {k3}")
    print(f"thực = k1 - k3 = {k1 - k3} | ảo = k1 + k2 = {k1 + k2}")
    return f"{k1 - k3}+{k1 + k2}i"

def trace_matrix(num1, num2):
    a, b = map(int, num1[:-1].split("+"))
    c, d = map(int, num2[:-1].split("+"))
    A, B = [[a, -b], [b, a]], [[c, -d], [d, c]]
    print("A =", A, "| B =", B)
    R = [[0, 0], [0, 0]]
    for i in range(2):
        for j in range(2):
            terms = [f"{A[i][k]}*{B[k][j]}" for k in range(2)]
            R[i][j] = sum(A[i][k] * B[k][j] for k in range(2))
            print(f"  R[{i}][{j}] = " + " + ".join(terms) + f" = {R[i][j]}")
    return f"{R[0][0]}+{R[1][0]}i"

print(trace_gauss("3+2i", "1+-4i"))
print(trace_matrix("3+2i", "1+-4i"))

k1 = c(a+b) = 1*(3+2) = 5
k2 = a(d-c) = 3*(-4-1) = -15
k3 = b(c+d) = 2*(1+-4) = -6
thực = k1 - k3 = 11 | ảo = k1 + k2 = -10
11+-10i
A = [[3, -2], [2, 3]] | B = [[1, 4], [-4, 1]]
  R[0][0] = 3*1 + -2*-4 = 11
  R[0][1] = 3*4 + -2*1 = 10
  R[1][0] = 2*1 + 3*-4 = -10
  R[1][1] = 2*4 + 3*1 = 11
11+-10i


Cả ba cách ra cùng một kết quả `"11+-10i"`. Ở cách 4, ta thấy `R[0][1] = -R[1][0]` và `R[0][0] = R[1][1]` nên `R` đúng dạng ma trận của một số phức.

## 9. Code nộp LeetCode

Dùng **cách 2** (công thức trực tiếp): ngắn nhất, nhanh nhất, dễ đọc nhất.

```python
class Solution:
    def complexNumberMultiply(self, num1: str, num2: str) -> str:
        a, b = map(int, num1[:-1].split("+"))
        c, d = map(int, num2[:-1].split("+"))
        return f"{a * c - b * d}+{a * d + b * c}i"
```

Chạy lại đúng đoạn code trên:

In [24]:
class Solution:
    def complexNumberMultiply(self, num1: str, num2: str) -> str:
        a, b = map(int, num1[:-1].split("+"))
        c, d = map(int, num2[:-1].split("+"))
        return f"{a * c - b * d}+{a * d + b * c}i"

# Mô phỏng cách LeetCode gọi
for (x, y), expected in examples:
    got = Solution().complexNumberMultiply(x, y)
    print(repr(x), repr(y), "->", repr(got), "OK" if got == expected else "SAI")
    assert got == expected

'1+1i' '1+1i' -> '0+2i' OK
'1+-1i' '1+-1i' -> '0+-2i' OK


## 10. Vì sao thuật toán đúng

**Bước đọc.** Theo đề, chuỗi hợp lệ có dạng `"<thực>+<ảo>i"`: đúng một dấu `+` (ở giữa) và đúng một chữ `i` ở **cuối**. Bỏ ký tự cuối bằng `[:-1]` còn `"<thực>+<ảo>"`; `split("+")` cắt tại dấu `+` duy nhất nên cho đúng hai chuỗi, mỗi chuỗi là số nguyên (có thể có dấu `-` ở đầu, nhưng dấu `-` không bị tách vì ta chỉ tách tại `+`). Do đó `int` nhận được đúng `a, b`.

**Bước tính.** Gọi `z1 = a + bi`, `z2 = c + di`. Theo tính phân phối của phép nhân trên các số phức (và `i^2 = -1`):

```
z1 * z2 = ac + adi + bci + bdi^2 = (ac - bd) + (ad + bc)i
```

nên `real = ac - bd` và `imag = ad + bc`. Đây chính xác là định nghĩa của tích hai số phức.

**Bước ghi.** `f"{real}+{imag}i"` ra đúng định dạng đề: phần thực, dấu `+`, phần ảo (số âm tự có dấu `-` đứng trước), chữ `i`. Không cần xử lý `-0` vì số nguyên Python không có "âm không".

**Tại sao các cách khác cũng đúng.**

- Cách 1: tương đương khai triển trên, chỉ khác là `i^k` được rút gọn tổng quát bằng chu kỳ 4.
- Cách 3: `k1 - k3 = ac - bd` và `k1 + k2 = ad + bc` (kiểm tra đại số ở mục 7).
- Cách 4: `M(z1) M(z2) = M(z1 z2)` (đã nhân ma trận ở mục 7).
- Cách 5: phép nhân `complex` của Python áp dụng đúng công thức trên; do mọi giá trị là số nguyên không quá `20000` nên float biểu diễn chính xác.

**Kiểm tra độc lập (tính chất).** Môđun bình phương `|z|^2 = a^2 + b^2` có tính **nhân**: `|z1 z2|^2 = |z1|^2 |z2|^2`. Ta dùng tính chất này như một phép kiểm tra không phụ thuộc công thức trong mục 12.

## 11. Phân tích Time / Space Complexity

Đầu vào chỉ là hai chuỗi có độ dài bị chặn (tối đa `4 + 1 + 4 + 1 = 10` ký tự).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đa thức + rút gọn `i^k` | `O(1)` | `O(1)` | 4 tích, từ điển nhỏ |
| 2. Công thức trực tiếp | `O(1)` | `O(1)` | 4 phép nhân |
| 3. Gauss | `O(1)` | `O(1)` | 3 phép nhân |
| 4. Ma trận `2 x 2` | `O(1)` | `O(1)` | 8 phép nhân |
| 5. `complex` + regex | `O(1)` | `O(1)` | regex tốn hằng số lớn hơn |

Tất cả là **hằng số**. Nếu tổng quát hóa với số nguyên cực lớn có `n` chữ số thì việc nhân hai số `n` chữ số mới chiếm thời gian: ở đó cách 3 (3 phép nhân) cho lợi ích thực sự (ý tưởng của thuật toán Karatsuba).

## 12. Bộ test

### 12.1. Ví dụ của đề và các ví dụ tự đặt

Tập hợp mọi cách vào một từ điển để chạy cùng bộ test:

In [25]:
METHODS = {
    "1-da-thuc": Solution1().complexNumberMultiply,
    "2-cong-thuc": Solution2().complexNumberMultiply,
    "3-gauss": Solution3().complexNumberMultiply,
    "4-ma-tran": Solution4().complexNumberMultiply,
    "5-complex": Solution5().complexNumberMultiply,
    "nop": Solution().complexNumberMultiply,
}

def check_all(num1, num2, expected):
    for name, fn in METHODS.items():
        got = fn(num1, num2)
        assert got == expected, (name, num1, num2, got, expected)

check_all("1+1i", "1+1i", "0+2i")
check_all("1+-1i", "1+-1i", "0+-2i")
check_all("3+2i", "1+-4i", "11+-10i")      # tính tay ở mục 4
print("ví dụ của đề và ví dụ tính tay: OK")

ví dụ của đề và ví dụ tính tay: OK


### 12.2. Edge case

- Có số `0`: `"0+0i"` nhân bất kỳ số nào ra `"0+0i"` (không có `-0`).
- Thuần thực (`"5+0i"`), thuần ảo (`"0+3i"`): `i * i = -1` nên `"0+1i" * "0+1i" = "-1+0i"`.
- Nhân với `"1+0i"` (phần tử đơn vị) giữ nguyên số kia.
- Giá trị biên `100` và `-100`: kết quả lớn nhất cỡ `20000`.
- Số liên hợp: `(a+bi)(a-bi) = a^2 + b^2` (phần ảo `0`).

In [26]:
edge = [
    ("0+0i", "0+0i", "0+0i"),
    ("0+0i", "5+-7i", "0+0i"),
    ("5+0i", "7+0i", "35+0i"),
    ("0+3i", "0+4i", "-12+0i"),
    ("0+1i", "0+1i", "-1+0i"),            # i * i = -1
    ("0+1i", "0+-1i", "1+0i"),            # i * (-i) = 1
    ("1+0i", "-37+58i", "-37+58i"),       # phần tử đơn vị
    ("100+100i", "100+100i", "0+20000i"),
    ("-100+-100i", "100+100i", "0+-20000i"),
    ("100+100i", "100+-100i", "20000+0i"),
    ("-100+100i", "-100+100i", "0+-20000i"),
    ("3+4i", "3+-4i", "25+0i"),           # liên hợp: a^2 + b^2
    ("-100+-100i", "-100+-100i", "0+20000i"),
]
for x, y, e in edge:
    check_all(x, y, e)
    print(f"{x:>11} * {y:<11} = {e}")
print("edge case: OK")

       0+0i * 0+0i        = 0+0i
       0+0i * 5+-7i       = 0+0i
       5+0i * 7+0i        = 35+0i
       0+3i * 0+4i        = -12+0i
       0+1i * 0+1i        = -1+0i
       0+1i * 0+-1i       = 1+0i
       1+0i * -37+58i     = -37+58i
   100+100i * 100+100i    = 0+20000i
 -100+-100i * 100+100i    = 0+-20000i
   100+100i * 100+-100i   = 20000+0i
  -100+100i * -100+100i   = 0+-20000i
       3+4i * 3+-4i       = 25+0i
 -100+-100i * -100+-100i  = 0+20000i
edge case: OK


### 12.3. Phản ví dụ: input làm các cách "gần đúng" sai

Các cặp dưới đây có **dấu âm** ở phần thực và/hoặc phần ảo, dễ làm lộ lỗi dấu (mục 13). Kết quả kỳ vọng được tính bằng hai nguồn độc lập: kiểu `complex` của Python (với số nguyên) và công thức viết tay.

In [27]:
def oracle_complex(num1, num2):
    """Oracle độc lập: dùng kiểu complex có sẵn, nhưng tự tách chuỗi bằng cách khác (replace)."""
    def to_c(s):
        real, imag = s.replace("i", "").split("+")
        return complex(int(real), int(imag))
    z = to_c(num1) * to_c(num2)
    return f"{int(z.real)}+{int(z.imag)}i"

tricky = [
    ("-1+-1i", "-1+-1i"),      # cả hai âm: thực = 1 - 1 = 0, ảo = 1 + 1 = 2
    ("2+-3i", "-4+5i"),
    ("-100+1i", "1+-100i"),
    ("0+-2i", "0+-2i"),        # (-2i)^2 = -4
    ("-7+0i", "0+-9i"),
]
for x, y in tricky:
    e = oracle_complex(x, y)
    check_all(x, y, e)
    print(f"{x:>8} * {y:<8} = {e}")

  -1+-1i * -1+-1i   = 0+2i
   2+-3i * -4+5i    = 7+22i
 -100+1i * 1+-100i  = 0+10001i
   0+-2i * 0+-2i    = -4+0i
   -7+0i * 0+-9i    = 0+63i


### 12.4. Kiểm tra vét cạn trên lưới nhỏ và tính chất đại số

Lưới `a, b, c, d` chạy trong `[-4, 4]` (`9^4 = 6561` cặp), đối chiếu với oracle `complex`:

In [28]:
count = 0
for a in range(-4, 5):
    for b in range(-4, 5):
        for c in range(-4, 5):
            for d in range(-4, 5):
                x, y = f"{a}+{b}i", f"{c}+{d}i"
                e = oracle_complex(x, y)
                for name, fn in METHODS.items():
                    assert fn(x, y) == e, (name, x, y)
                count += 1
print("vét cạn", count, "cặp x", len(METHODS), "cách: OK")

vét cạn 6561 cặp x 6 cách: OK


In [29]:
def parse_pair(s):
    a, b = s[:-1].split("+")
    return int(a), int(b)

def norm_sq(s):
    a, b = parse_pair(s)
    return a * a + b * b

rng = random.Random(537)
sol = Solution().complexNumberMultiply
for _ in range(2000):
    x = f"{rng.randint(-100, 100)}+{rng.randint(-100, 100)}i"
    y = f"{rng.randint(-100, 100)}+{rng.randint(-100, 100)}i"
    z = f"{rng.randint(-100, 100)}+{rng.randint(-100, 100)}i"
    xy = sol(x, y)
    # tính chất 1: giao hoán
    assert xy == sol(y, x)
    # tính chất 2: môđun bình phương có tính nhân (kiểm tra độc lập với công thức)
    assert norm_sq(xy) == norm_sq(x) * norm_sq(y)
print("giao hoán và tính nhân của môđun: OK")

giao hoán và tính nhân của môđun: OK


Tính chất kết hợp `(xy)z = x(yz)` cần nhân tích (có thể lớn hơn `100`) với số khác. Hàm `Solution` vẫn tính đúng vì chỉ dùng số nguyên Python (không giới hạn), nên ta kiểm tra riêng:

In [30]:
def mul_str(x, y):
    return Solution().complexNumberMultiply(x, y)

rng = random.Random(5370)
for _ in range(2000):
    x = f"{rng.randint(-100, 100)}+{rng.randint(-100, 100)}i"
    y = f"{rng.randint(-100, 100)}+{rng.randint(-100, 100)}i"
    z = f"{rng.randint(-100, 100)}+{rng.randint(-100, 100)}i"
    assert mul_str(mul_str(x, y), z) == mul_str(x, mul_str(y, z))
    # phân phối: x(y + z) = xy + xz, cộng bằng tay trên cặp số
    yp, zp = parse_pair(y), parse_pair(z)
    s = f"{yp[0] + zp[0]}+{yp[1] + zp[1]}i"
    p1, p2, p3 = map(parse_pair, (mul_str(x, s), mul_str(x, y), mul_str(x, z)))
    assert p1 == (p2[0] + p3[0], p2[1] + p3[1])
print("kết hợp và phân phối: OK")

kết hợp và phân phối: OK


### 12.5. Đối chiếu ngẫu nhiên đúng ràng buộc (seed cố định) và hiệu năng

Sinh `a, b, c, d` ngẫu nhiên trong `[-100, 100]` (seed `537`), so **tất cả** các cách với oracle. Sau đó đo thời gian của từng cách cho `50000` phép nhân (để thấy cả năm đều rất nhanh; con số chỉ mang tính tham khảo).

In [31]:
rng = random.Random(537)
cases = []
for _ in range(3000):
    x = f"{rng.randint(-100, 100)}+{rng.randint(-100, 100)}i"
    y = f"{rng.randint(-100, 100)}+{rng.randint(-100, 100)}i"
    cases.append((x, y))
for x, y in cases:
    e = oracle_complex(x, y)
    for name, fn in METHODS.items():
        assert fn(x, y) == e, (name, x, y)
print("3000 cặp ngẫu nhiên x", len(METHODS), "cách: OK")

batch = [cases[i % len(cases)] for i in range(50000)]
for name, fn in METHODS.items():
    t0 = time.perf_counter()
    for x, y in batch:
        fn(x, y)
    print(f"{name:12} {(time.perf_counter() - t0) * 1000:8.1f} ms / 50000 phép nhân")

3000 cặp ngẫu nhiên x 6 cách: OK


1-da-thuc       227.2 ms / 50000 phép nhân
2-cong-thuc      98.4 ms / 50000 phép nhân
3-gauss         100.6 ms / 50000 phép nhân


4-ma-tran       312.4 ms / 50000 phép nhân
5-complex       118.9 ms / 50000 phép nhân
nop              57.5 ms / 50000 phép nhân


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `i^2 = -1` (cộng `bd` thay vì trừ)

**Code sai:**

In [32]:
def wrong_forgot_i_squared(num1, num2):
    a, b = map(int, num1[:-1].split("+"))
    c, d = map(int, num2[:-1].split("+"))
    return f"{a * c + b * d}+{a * d + b * c}i"      # SAI: phần thực phải là ac - bd

print("sai :", wrong_forgot_i_squared("1+1i", "1+1i"), "(kỳ vọng 0+2i)")
print("đúng:", Solution().complexNumberMultiply("1+1i", "1+1i"))

sai : 2+2i (kỳ vọng 0+2i)
đúng: 0+2i


**Vì sao sai:** số hạng `bi * di = bd * i^2 = -bd` đóng góp vào phần **thực** với dấu **âm**. Cộng `bd` làm `(1+i)^2` ra `2+2i` thay vì `2i`.

**Code đúng:** `real = a*c - b*d`.

### Lỗi 2: Quên bỏ chữ `i` cuối trước khi `int`

In [33]:
def wrong_no_strip(num1, num2):
    a, b = map(int, num1.split("+"))                # SAI: "1i" không phải số nguyên
    c, d = map(int, num2.split("+"))
    return f"{a * c - b * d}+{a * d + b * c}i"

try:
    wrong_no_strip("1+1i", "1+1i")
except ValueError as e:
    print("sai : ValueError:", e)
print("đúng:", Solution().complexNumberMultiply("1+1i", "1+1i"))

sai : ValueError: invalid literal for int() with base 10: '1i'
đúng: 0+2i


**Vì sao sai:** sau `split("+")`, phần tử thứ hai là `"1i"`; `int("1i")` báo lỗi. Phải bỏ chữ `i` (bằng `s[:-1]` hoặc `rstrip("i")`) **trước** khi tách.

**Code đúng:** `num[:-1].split("+")`.

### Lỗi 3: Định dạng đầu ra sai khi phần ảo âm

In [34]:
def wrong_format(num1, num2):
    a, b = map(int, num1[:-1].split("+"))
    c, d = map(int, num2[:-1].split("+"))
    real, imag = a * c - b * d, a * d + b * c
    sign = "+" if imag >= 0 else ""                  # SAI: tự "tiện tay" bỏ dấu + khi âm
    return f"{real}{sign}{imag}i"

print("sai :", wrong_format("1+-1i", "1+-1i"), "(đề yêu cầu 0+-2i)")
print("đúng:", Solution().complexNumberMultiply("1+-1i", "1+-1i"))

sai : 0-2i (đề yêu cầu 0+-2i)
đúng: 0+-2i


**Vì sao sai:** đề quy định dạng `real+imaginaryi` **luôn** có dấu `+`, kể cả khi phần ảo âm (`"0+-2i"`). Viết `"0-2i"` là sai định dạng và bị chấm sai.

**Code đúng:** `f"{real}+{imag}i"`, không xử lý dấu riêng.

### Lỗi 4: Dùng số thực/hình học cực (float) làm sai kết quả

Một ý tưởng hấp dẫn: số phức có **môđun** `r` và **góc** `theta`; nhân hai số phức = nhân môđun và **cộng góc**. Nhưng `sqrt`, `atan2`, `cos`, `sin` cho kết quả float **có sai số**:

In [35]:
def wrong_polar(num1, num2):
    a, b = map(int, num1[:-1].split("+"))
    c, d = map(int, num2[:-1].split("+"))
    r1, t1 = cmath.polar(complex(a, b))
    r2, t2 = cmath.polar(complex(c, d))
    z = cmath.rect(r1 * r2, t1 + t2)
    return f"{int(z.real)}+{int(z.imag)}i"                  # SAI: int() cắt cụt phần thập phân

first_bad = None
for a in range(-5, 6):
    for b in range(-5, 6):
        for c in range(-5, 6):
            for d in range(-5, 6):
                x, y = f"{a}+{b}i", f"{c}+{d}i"
                if wrong_polar(x, y) != Solution().complexNumberMultiply(x, y):
                    first_bad = (x, y)
                    break
            if first_bad: break
        if first_bad: break
    if first_bad: break

x, y = first_bad
print("ví dụ sai:", x, "*", y)
print("sai (cực):", wrong_polar(x, y))
print("đúng     :", Solution().complexNumberMultiply(x, y))

bad = sum(
    wrong_polar(f"{a}+{b}i", f"{c}+{d}i") != Solution().complexNumberMultiply(f"{a}+{b}i", f"{c}+{d}i")
    for a in range(-5, 6) for b in range(-5, 6) for c in range(-5, 6) for d in range(-5, 6)
)
print("số cặp sai trên lưới 11^4 =", 14641, ":", bad)

ví dụ sai: -5+-5i * -5+-3i
sai (cực): 9+40i
đúng     : 10+40i
số cặp sai trên lưới 11^4 = 14641 : 7240


**Vì sao sai:** kết quả thật là số nguyên, nhưng số thực lại ra dạng `9.999999999999998` và `int()` **cắt cụt** thành `9`. (Dùng `round` có thể cứu được ở đây, nhưng không cần thiết vì đã có công thức nguyên chính xác.)

**Code đúng:** làm toàn bộ bằng **số nguyên**: công thức `(ac - bd) + (ad + bc)i`.

### Lỗi 5: Đưa thẳng chuỗi vào `complex(...)`

In [36]:
try:
    complex("1+-1j")
except ValueError as e:
    print("sai : complex('1+-1j') -> ValueError:", e)

try:
    complex("1+1i")
except ValueError as e:
    print("sai : complex('1+1i')  -> ValueError:", e)
print("đúng: tách bằng split rồi complex(a, b):", complex(1, -1))

sai : complex('1+-1j') -> ValueError: complex() arg is a malformed string
sai : complex('1+1i')  -> ValueError: complex() arg is a malformed string
đúng: tách bằng split rồi complex(a, b): (1-1j)


**Vì sao sai:** `complex(chuỗi)` chỉ chấp nhận chuẩn của Python: chữ `j` (không phải `i`) và **không** cho hai dấu liền nhau (`+-`). Chuỗi của đề không đúng chuẩn đó.

**Code đúng:** tự tách `a`, `b` thành số nguyên rồi dùng `complex(a, b)` (cách 5) hoặc dùng công thức.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nhận dạng:** bài "mô phỏng" (Simulation) với **định dạng chuỗi cố định**: phần lớn công sức là **đọc đúng và ghi đúng**, phần toán rất ngắn.
- Công thức nên thuộc: `(a+bi)(c+di) = (ac - bd) + (ad + bc)i`. Cách nhớ: "thực = thực*thực **trừ** ảo*ảo; ảo = chéo cộng chéo".
- Mẹo đọc chuỗi: `s[:-1].split("+")`; dùng `map(int, ...)` để đổi cả hai nửa.
- Định dạng ra: `f"{real}+{imag}i"`; không tự thêm bớt dấu.
- Nếu đề **cho phép** viết `"a-bi"` hoặc bỏ hệ số `1` (ví dụ `"i"`, `"3-i"`), cần bộ đọc tổng quát bằng regex (mục 15).
- **Kiểm tra nhanh bằng tính chất:** `|z1 z2|^2 = |z1|^2 |z2|^2` và `i * i = -1`; so với `complex` của Python bằng số nguyên nhỏ.
- Bài họ hàng: LeetCode 592 (Fraction Addition and Subtraction: đọc biểu thức dạng chuỗi rồi tính), 224/227 (Basic Calculator), 43 (Multiply Strings), 8 (String to Integer), 640 (Solve the Equation), 166 (Fraction to Recurring Decimal).

## 15. Biến thể

### 15.1. Đọc số phức dạng tổng quát (`"3-4i"`, `"-i"`, `"5"`, `"2i"`)

Nếu đề cho phép nhiều cách viết, cần một bộ đọc tổng quát hơn: phần thực tùy chọn, phần ảo tùy chọn, hệ số `1` có thể bị lược. Dưới đây là một bộ đọc nhỏ cho số nguyên (tìm dấu `+`/`-` cuối cùng để tách phần ảo).

In [37]:
def parse_general(s):
    """'3-4i' -> (3, -4); '-i' -> (0, -1); '5' -> (5, 0); '2i' -> (0, 2); '1+i' -> (1, 1)"""
    s = s.replace(" ", "").replace("+-", "-")     # "1+-1i" -> "1-1i": chuẩn hóa về một dấu
    # tách phần ảo: kết thúc bằng 'i'
    if not s.endswith("i"):
        return int(s), 0
    body = s[:-1]                        # bỏ chữ i
    # tìm dấu +/- bắt đầu phần ảo (không tính ký tự đầu tiên)
    k = max(body.rfind("+"), body.rfind("-"))
    if k <= 0:                           # chỉ có phần ảo
        real, imag_s = 0, body
    else:
        real, imag_s = int(body[:k]), body[k:]
    imag = {"": 1, "+": 1, "-": -1}.get(imag_s, None)
    if imag is None:
        imag = int(imag_s)
    return real, imag

for s in ["3-4i", "-i", "5", "2i", "1+i", "-3+-2i", "0+0i", "-7+5i", "i"]:
    print(f"{s:8} -> {parse_general(s)}")
assert parse_general("1+-1i") == (1, -1)

3-4i     -> (3, -4)
-i       -> (0, -1)
5        -> (5, 0)
2i       -> (0, 2)
1+i      -> (1, 1)
-3+-2i   -> (-3, -2)
0+0i     -> (0, 0)
-7+5i    -> (-7, 5)
i        -> (0, 1)


Cách đọc tổng quát cho thấy vì sao đề cố định định dạng `a+bi` là một ưu ái: với `"1+-1i"` chỉ cần `split("+")`.

### 15.2. Phép chia số phức (dùng số liên hợp)

`z1 / z2 = z1 * conj(z2) / |z2|^2` trong đó `conj(c + di) = c - di` và `|z2|^2 = c^2 + d^2`. Kết quả là **số hữu tỉ**, ta dùng `Fraction` để chính xác:

In [38]:
def divide(num1, num2):
    a, b = parse_pair(num1)
    c, d = parse_pair(num2)
    den = c * c + d * d
    assert den != 0, "chia cho 0"
    real = Fraction(a * c + b * d, den)       # z1 * conj(z2): (a + bi)(c - di)
    imag = Fraction(b * c - a * d, den)
    return real, imag

print(divide("1+1i", "1+-1i"))        # (1+i)/(1-i) = i
print(divide("3+2i", "1+-4i"))
# kiểm tra: (z1 / z2) * z2 = z1
r, i = divide("3+2i", "1+-4i")
c, d = parse_pair("1+-4i")
print((r * c - i * d, r * d + i * c))

(Fraction(0, 1), Fraction(1, 1))
(Fraction(-5, 17), Fraction(14, 17))
(Fraction(3, 1), Fraction(2, 1))


### 15.3. Lũy thừa `z^k` bằng bình phương nhanh (square-and-multiply)

`z^k` cần `O(log k)` phép nhân số phức thay vì `k`. Mỗi lần nhân dùng công thức của bài. (Chú ý: giá trị lớn dần; Python có số nguyên không giới hạn nên không tràn.)

In [39]:
def complex_pow(num, k):
    result = "1+0i"
    base = num
    while k > 0:
        if k & 1:
            result = Solution().complexNumberMultiply(result, base)
        base = Solution().complexNumberMultiply(base, base)
        k >>= 1
    return result

print(complex_pow("0+1i", 2), complex_pow("0+1i", 4), complex_pow("1+1i", 8))   # -1+0i, 1+0i, 16+0i
# đối chiếu với vòng lặp nhân thẳng
cur = "1+0i"
for k in range(0, 13):
    assert complex_pow("1+2i", k) == cur
    cur = Solution().complexNumberMultiply(cur, "1+2i")
print("complex_pow khớp phép nhân lặp: OK")

-1+0i 1+0i 16+0i
complex_pow khớp phép nhân lặp: OK


### 15.4. Cộng và trừ số phức (cùng định dạng)

Chỉ cần cộng/trừ **từng phần**: `(a + bi) + (c + di) = (a + c) + (b + d)i`.

In [40]:
def add(num1, num2):
    (a, b), (c, d) = parse_pair(num1), parse_pair(num2)
    return f"{a + c}+{b + d}i"

def sub(num1, num2):
    (a, b), (c, d) = parse_pair(num1), parse_pair(num2)
    return f"{a - c}+{b - d}i"

print(add("1+2i", "3+-5i"), sub("1+2i", "3+-5i"))

4+-3i -2+7i


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đa thức + rút gọn `i^k` | nhân hai "ngoặc" tổng quát, rút gọn bằng `k mod 4` | `O(1)` | `O(1)` | khi quên công thức hoặc mở rộng lên đa thức |
| 2. Công thức | `(ac-bd) + (ad+bc)i` | `O(1)` | `O(1)` | **nộp bài**: ngắn, rõ, nhanh |
| 3. Gauss | 3 phép nhân thay vì 4 | `O(1)` | `O(1)` | số rất lớn, hoặc muốn biết mẹo Karatsuba |
| 4. Ma trận `2 x 2` | nhân ma trận `[[a,-b],[b,a]]` | `O(1)` | `O(1)` | học góc nhìn đại số, quay trong mặt phẳng |
| 5. `complex` + regex | dùng kiểu số phức có sẵn | `O(1)` | `O(1)` | prototype nhanh; cẩn thận float |

Kết luận: chọn **cách 2** khi nộp. Các cách còn lại có giá trị học tập: cách 1 và 4 giúp **hiểu** vì sao công thức đúng, cách 3 cho thấy có thể tối ưu số phép nhân, cách 5 cho thấy thư viện làm sẵn nhưng phải cẩn thận kiểu số.

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán đầu ra của ba dòng sau rồi chạy:

In [41]:
print(Solution().complexNumberMultiply("0+1i", "0+1i"))
print(Solution().complexNumberMultiply("2+0i", "-3+4i"))
print(Solution().complexNumberMultiply("1+1i", "1+-1i"))

-1+0i
-6+8i
2+0i


**Bài 2 (tính tay).** Tính `(2 + 3i)(4 + -5i)` bằng bảng nhân (mục 4) và gán kết quả dạng chuỗi (`"thực+ảoi"`) vào `ans2` (đang là `None`).

In [42]:
ans2 = None      # TODO: điền chuỗi, ví dụ "7+-1i"

**Bài 3 (điền TODO).** Viết `parse_num(s)` trả về **tuple** `(thực, ảo)` gồm hai số nguyên từ chuỗi dạng `"a+bi"`. Ví dụ `"1+-1i"` -> `(1, -1)`.

In [43]:
def parse_num(s):
    # TODO: bỏ chữ i cuối, split("+"), đổi sang int
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_multiply(num1, num2)` theo **mẹo Gauss** (cách 3: `k1, k2, k3`).

In [44]:
def my_multiply(num1, num2):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `conjugate(num)` trả về số liên hợp của `"a+bi"` là `"a+-bi"` viết lại đúng định dạng (chú ý phần ảo `0` phải ra `"a+0i"` chứ không phải `"a+-0i"`). (b) Viết `power(num, k)` tính `num^k` (với `k >= 0`) bằng **bình phương nhanh**, trả về chuỗi; `power(num, 0) = "1+0i"`.

In [45]:
def conjugate(num):
    # TODO
    pass

def power(num, k):
    # TODO: square-and-multiply, dùng Solution().complexNumberMultiply
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: dùng `i * i = -1` và công thức `(ac - bd) + (ad + bc)i`.
- Bài 2: lập bảng 2 x 2: `ac`, `adi`, `bci`, `bdi^2`.
- Bài 3: dùng ví dụ nhỏ ở mục 3.3 và 3.4.
- Bài 4: nhớ 3 biến `k1 = c(a+b)`, `k2 = a(d-c)`, `k3 = b(c+d)`.
- Bài 5: `-b` khi `b = 0` là `0` (số nguyên Python không có `-0`).

**Mức 2 (vừa):**
- Bài 2: `a=2, b=3, c=4, d=-5`: `ac = 8`, `bd = -15`, `ad = -10`, `bc = 12`.
- Bài 3: `real, imag = s[:-1].split("+")` rồi `return int(real), int(imag)`.
- Bài 4: `real = k1 - k3`, `imag = k1 + k2`; ghép bằng f-string.
- Bài 5: `conjugate` = đổi dấu phần ảo; `power`: `while k > 0:` nếu `k` lẻ thì nhân `result` với `base`; rồi bình phương `base`, `k >>= 1`.

**Mức 3 (gần đáp án):**
- Bài 2: thực `= ac - bd = 8 + 15 = 23`; ảo `= ad + bc = -10 + 12 = 2` -> `"23+2i"`.
- Bài 3: `def parse_num(s): r, i = s[:-1].split("+"); return int(r), int(i)`.
- Bài 5: `conjugate`: `a, b = parse_num(num); return f"{a}+{-b}i"`; `power`: khởi tạo `result = "1+0i"`, `base = num`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [46]:
# Bài 1: "-1+0i", "-6+8i", "2+0i"
assert Solution().complexNumberMultiply("0+1i", "0+1i") == "-1+0i"
assert Solution().complexNumberMultiply("2+0i", "-3+4i") == "-6+8i"
assert Solution().complexNumberMultiply("1+1i", "1+-1i") == "2+0i"

# Bài 2: (2+3i)(4-5i) = 8 - 10i + 12i - 15i^2 = 8 + 15 + 2i = 23 + 2i
ans2 = "23+2i"
assert ans2 == Solution().complexNumberMultiply("2+3i", "4+-5i")
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [47]:
# Bài 3
def parse_num(s):
    real, imag = s[:-1].split("+")
    return int(real), int(imag)

assert parse_num("1+-1i") == (1, -1)
assert parse_num("-100+100i") == (-100, 100)
assert parse_num("0+0i") == (0, 0)
print("Bài 3 OK")

Bài 3 OK


In [48]:
# Bài 4
def my_multiply(num1, num2):
    a, b = parse_num(num1)
    c, d = parse_num(num2)
    k1 = c * (a + b)
    k2 = a * (d - c)
    k3 = b * (c + d)
    return f"{k1 - k3}+{k1 + k2}i"

rng = random.Random(17)
for _ in range(500):
    x = f"{rng.randint(-100, 100)}+{rng.randint(-100, 100)}i"
    y = f"{rng.randint(-100, 100)}+{rng.randint(-100, 100)}i"
    assert my_multiply(x, y) == Solution().complexNumberMultiply(x, y)
print("Bài 4 OK")

Bài 4 OK


In [49]:
# Bài 5
def conjugate(num):
    a, b = parse_num(num)
    return f"{a}+{-b}i"

def power(num, k):
    result, base = "1+0i", num
    while k > 0:
        if k & 1:
            result = Solution().complexNumberMultiply(result, base)
        base = Solution().complexNumberMultiply(base, base)
        k >>= 1
    return result

assert conjugate("3+4i") == "3+-4i"
assert conjugate("3+-4i") == "3+4i"
assert conjugate("5+0i") == "5+0i"             # không phải "5+-0i"
assert power("1+1i", 0) == "1+0i"
assert power("0+1i", 3) == "0+-1i"             # i^3 = -i
assert power("1+1i", 8) == "16+0i"
# (z * conj z) = |z|^2
for s in ["3+4i", "-2+7i", "0+5i"]:
    a, b = parse_num(s)
    assert Solution().complexNumberMultiply(s, conjugate(s)) == f"{a * a + b * b}+0i"
cur = "1+0i"
for k in range(10):
    assert power("2+-3i", k) == cur
    cur = Solution().complexNumberMultiply(cur, "2+-3i")
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (cách 2):**

```python
class Solution:
    def complexNumberMultiply(self, num1: str, num2: str) -> str:
        a, b = map(int, num1[:-1].split("+"))     # bỏ 'i' cuối, tách theo '+'
        c, d = map(int, num2[:-1].split("+"))
        return f"{a * c - b * d}+{a * d + b * c}i"   # (ac-bd) + (ad+bc)i
```

**Mẹo Gauss (3 phép nhân):**

```python
k1 = c * (a + b); k2 = a * (d - c); k3 = b * (c + d)
real, imag = k1 - k3, k1 + k2
```

**Checklist trước khi submit:**

- [ ] Đã **bỏ chữ `i` cuối** trước khi `int` (`s[:-1]`)?
- [ ] Tách theo `"+"` (dấu `-` chỉ là dấu âm trong từng số, không tách)?
- [ ] Phần thực là `ac - bd` (**trừ**, vì `i^2 = -1`) và phần ảo là `ad + bc`?
- [ ] Đầu ra có dạng `"thực+ảoi"` với dấu `+` **luôn có** (kể cả `"0+-2i"`)?
- [ ] Không dùng float / `sqrt` / `cos` / `sin` (dùng số nguyên cho chính xác)?
- [ ] Không dùng thẳng `complex("...")` với chuỗi của đề (chữ `i`, hai dấu liền nhau)?
- [ ] Đã thử: có số `0`, thuần ảo (`i*i`), số âm cả hai, biên `100` và `-100`?